# Contoh Simulasi WAW (Write After Write)

Jalankan sel di bawah (Runtime → Run all).

In [2]:
# Simulator from https://github.com/harshitasingh4863/instruction-level-parallelism-simulator.git
# =================================================================
# CONTOH SIMULASI WAW (Write After Write) / Output Dependency
#   WAW terjadi jika instruksi j MENULIS register yang juga DITULIS
#   oleh instruksi i sebelumnya (i < j). Urutan penulisan harus
#   dijaga agar nilai akhir register berasal dari instruksi terakhir.
#   Contoh:  I1: R1 = R2 + R3
#            I3: R1 = R7 + R8   <- I3 menulis ulang R1 milik I1 (WAW)
#   WAW hanya konflik nama register, sehingga bisa dihilangkan
#   dengan register renaming (mis. I3 menulis ke R21).
# =================================================================

from dataclasses import dataclass

@dataclass
class Instruction:
    name: str           # nama instruksi
    destination: str    # register yang ditulis
    sources: list       # register yang dibaca


# -------------------------------------------------
# Sample Instruction Set (contoh dengan dependensi WAW)
# -------------------------------------------------

instructions = [
    Instruction("I1", "R1", ["R2", "R3"]),     # menulis R1
    Instruction("I2", "R4", ["R5", "R6"]),     # menulis R4
    Instruction("I3", "R1", ["R7", "R8"]),     # WAW: menulis ulang R1 (setelah I1)
    Instruction("I4", "R9", ["R10", "R11"]),   # menulis R9
    Instruction("I5", "R4", ["R12", "R13"]),   # WAW: menulis ulang R4 (setelah I2)
    Instruction("I6", "R1", ["R14", "R15"]),   # WAW: menulis ulang R1 (setelah I1 dan I3)
    Instruction("I7", "R9", ["R16", "R17"]),   # WAW: menulis ulang R9 (setelah I4)
    Instruction("I8", "R18", ["R19", "R20"]),  # independen
]


# -------------------------------------------------
# Function to check dependency
# -------------------------------------------------

def has_dependency(inst1, inst2):
    """
    Returns True if inst2 depends on inst1.
    """

    # WAW dependency:
    # inst2 menulis register yang sama dengan inst1,
    # jadi inst2 harus menunggu inst1 selesai menulis
    if inst1.destination == inst2.destination:
        return True

    return False


# -------------------------------------------------
# Sequential Execution (1 instruksi per cycle)
# -------------------------------------------------

def sequential_execution(instructions):
    print("\n" + "=" * 50)
    print("SEQUENTIAL EXECUTION")
    print("=" * 50)

    cycle = 0

    for instruction in instructions:
        cycle += 1

        print(
            f"Cycle {cycle}: Executing {instruction.name} "
            f"({instruction.destination} = "
            f"{' + '.join(instruction.sources)})"
        )

    print(f"\nTotal Sequential Cycles: {cycle}")

    return cycle


# -------------------------------------------------
# ILP / Parallel Execution
# -------------------------------------------------

def parallel_execution(instructions):
    print("\n" + "=" * 50)
    print("INSTRUCTION-LEVEL PARALLEL EXECUTION")
    print("=" * 50)

    completed = set()
    remaining = instructions.copy()

    cycle = 0

    while remaining:

        cycle += 1
        current_cycle = []

        for instruction in remaining:

            # Instruksi hanya boleh jalan jika semua instruksi
            # sebelumnya yang menulis register tujuan yang sama (WAW)
            # sudah selesai, agar urutan penulisan terjaga

            dependencies_satisfied = True

            for previous in instructions:

                if previous.name == instruction.name:
                    break

                if has_dependency(previous, instruction):
                    if previous.name not in completed:
                        dependencies_satisfied = False
                        break

            if dependencies_satisfied:
                current_cycle.append(instruction)

        # Instruksi yang tidak menunggu WAW
        # dieksekusi bersamaan dalam cycle yang sama

        print(f"\nCycle {cycle}:")

        for instruction in current_cycle:
            print(
                f"  -> {instruction.name}: "
                f"{instruction.destination} = "
                f"{' + '.join(instruction.sources)}"
            )

        # Mark instructions as completed

        for instruction in current_cycle:
            completed.add(instruction.name)
            remaining.remove(instruction)

    print(f"\nTotal ILP Cycles: {cycle}")

    return cycle


# -------------------------------------------------
# Dependency Analysis
# -------------------------------------------------

def show_dependencies(instructions):

    print("\n" + "=" * 50)
    print("DEPENDENCY ANALYSIS (WAW)")
    print("=" * 50)

    found = False

    for i in range(len(instructions)):

        for j in range(i + 1, len(instructions)):

            if has_dependency(instructions[i], instructions[j]):

                print(
                    f"{instructions[j].name} depends on "
                    f"{instructions[i].name} "
                    f"(WAW, Register: {instructions[i].destination})"
                )

                found = True

    if not found:
        print("No dependencies found.")


# -------------------------------------------------
# Main Program
# -------------------------------------------------

def main():

    print("\n")
    print("*" * 60)
    print("   INSTRUCTION-LEVEL PARALLELISM SIMULATOR - WAW")
    print("*" * 60)

    print("\nInstruction Set:")

    for instruction in instructions:
        print(
            f"{instruction.name}: "
            f"{instruction.destination} = "
            f"{' + '.join(instruction.sources)}"
        )

    # Show dependencies

    show_dependencies(instructions)

    # Sequential execution

    sequential_cycles = sequential_execution(instructions)

    # Parallel execution

    parallel_cycles = parallel_execution(instructions)

    # Calculate speedup

    speedup = sequential_cycles / parallel_cycles

    efficiency = speedup / len(instructions) * 100

    print("\n" + "=" * 50)
    print("PERFORMANCE COMPARISON")
    print("=" * 50)

    print(f"Sequential Cycles : {sequential_cycles}")
    print(f"ILP Cycles        : {parallel_cycles}")
    print(f"Speedup            : {speedup:.2f}x")
    print(f"ILP Efficiency     : {efficiency:.2f}%")

    print("\nConclusion:")
    print(
        "WAW dependency forces writes to the same register to happen "
        "in program order (I1 -> I3 -> I6 for R1). Because it is only "
        "a register-name conflict, register renaming can remove it."
    )


# -------------------------------------------------
# Run Program
# -------------------------------------------------

if __name__ == "__main__":
    main()



************************************************************
   INSTRUCTION-LEVEL PARALLELISM SIMULATOR - WAW
************************************************************

Instruction Set:
I1: R1 = R2 + R3
I2: R4 = R5 + R6
I3: R1 = R7 + R8
I4: R9 = R10 + R11
I5: R4 = R12 + R13
I6: R1 = R14 + R15
I7: R9 = R16 + R17
I8: R18 = R19 + R20

DEPENDENCY ANALYSIS (WAW)
I3 depends on I1 (WAW, Register: R1)
I6 depends on I1 (WAW, Register: R1)
I5 depends on I2 (WAW, Register: R4)
I6 depends on I3 (WAW, Register: R1)
I7 depends on I4 (WAW, Register: R9)

SEQUENTIAL EXECUTION
Cycle 1: Executing I1 (R1 = R2 + R3)
Cycle 2: Executing I2 (R4 = R5 + R6)
Cycle 3: Executing I3 (R1 = R7 + R8)
Cycle 4: Executing I4 (R9 = R10 + R11)
Cycle 5: Executing I5 (R4 = R12 + R13)
Cycle 6: Executing I6 (R1 = R14 + R15)
Cycle 7: Executing I7 (R9 = R16 + R17)
Cycle 8: Executing I8 (R18 = R19 + R20)

Total Sequential Cycles: 8

INSTRUCTION-LEVEL PARALLEL EXECUTION

Cycle 1:
  -> I1: R1 = R2 + R3
  -> I2: R4 = R5 + R6
 